# 🎤 KaraokeAI Backend Server
### Run this notebook to start the backend API for your KaraokeAI app
**Steps:** Run each cell one by one from top to bottom.
- After the last cell, you will get a **public URL** — copy and paste it into your app!

In [10]:
# ── CELL 1: Install all dependencies ──────────────────────────
print('Installing dependencies... (takes 2-3 minutes)')
!pip install -q faster-whisper demucs ffmpeg-python pysubs2 fastapi uvicorn pyngrok nest-asyncio transliterate
!apt-get install -q ffmpeg
print('✅ All dependencies installed!')

Installing dependencies... (takes 2-3 minutes)
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
✅ All dependencies installed!


In [11]:
# ── CELL 2: Clone your GitHub repository ──────────────────────
import os

GITHUB_REPO = 'https://github.com/shanmuganathan457/karaoke-video-generator.git'

if os.path.exists('karaoke-video-generator'):
    print('Repo already exists, pulling latest changes...')
    !cd karaoke-video-generator && git pull
else:
    print('Cloning your GitHub repository...')
    !git clone {GITHUB_REPO}

os.chdir('karaoke-video-generator')
print(f'✅ Working directory: {os.getcwd()}')
print('📁 Files found:', os.listdir('.'))

Cloning your GitHub repository...
Cloning into 'karaoke-video-generator'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: 'karaoke-video-generator'

In [ ]:
# ── CELL 3: Write the FastAPI server ──────────────────────────
api_code = '''
import sys, os, shutil, uuid
sys.path.insert(0, os.getcwd())

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

from karaoke_generator.transcriber import AudioTranscriber
from karaoke_generator.subtitle_generator import SubtitleGenerator
from karaoke_generator.video_processor import VideoProcessor
from karaoke_generator.vocal_separator import VocalSeparator

app = FastAPI(title="KaraokeAI Backend API")

# Allow requests from your Vercel frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "KaraokeAI Backend is running!", "version": "1.0.0"}

@app.post("/generate")
async def generate_karaoke(
    video: UploadFile = File(...),
    language: str = Form(default=""),
    romanize: bool = Form(default=True)
):
    job_id = str(uuid.uuid4())[:8]
    input_path = f"/tmp/input_{job_id}.mp4"
    temp_audio = f"/tmp/audio_{job_id}.wav"
    output_ass  = f"/tmp/karaoke_{job_id}.ass"
    output_video = f"/tmp/output_{job_id}.mp4"
    vocals_audio = None
    instrumental_audio = None

    try:
        # Save uploaded file
        with open(input_path, "wb") as f:
            content = await video.read()
            f.write(content)

        processor = VideoProcessor()
        transcriber = AudioTranscriber()
        sub_gen = SubtitleGenerator()

        # Step 1: Extract Audio
        processor.extract_audio(input_path, temp_audio)

        # Step 2: Separate Vocals
        vocals_audio, instrumental_audio = VocalSeparator.separate(temp_audio)

        # Step 3: Transcribe
        result = transcriber.transcribe(
            vocals_audio,
            language=language if language else None,
            vad_filter=False
        )

        # Step 4: Generate Subtitles
        sub_gen.generate(result["segments"], output_ass, romanize=romanize, language=result["language"])

        # Step 5: Burn Subtitles
        processor.burn_subtitles(input_path, output_ass, output_video)

        return FileResponse(
            output_video,
            media_type="video/mp4",
            filename="karaoke_output.mp4"
        )

    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

    finally:
        for f in [input_path, temp_audio, vocals_audio, instrumental_audio, output_ass]:
            if f and os.path.exists(f):
                os.remove(f)
'''

with open('api_server.py', 'w') as f:
    f.write(api_code)

print('✅ API server file created!')

In [ ]:
# ── CELL 4: Enter your ngrok token ────────────────────────────
# Get your FREE token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below between the quotes:

NGROK_TOKEN = "3K3ZdIf4fOiZoKNgHMNkp1OMjzx_4m3KxC1tiARvKtNiZKKq7"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set!')

In [ ]:
# ── CELL 5: START THE SERVER ──────────────────────────────────
# This will print your public URL — copy it and paste into your React app!

import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok

nest_asyncio.apply()

# Open a tunnel on port 8000
public_url = ngrok.connect(8000)
print('=' * 60)
print('🚀 KaraokeAI Backend is LIVE!')
print('=' * 60)
print(f'\n📡 YOUR PUBLIC API URL:')
print(f'   {public_url}')
print(f'\n✅ Copy this URL and paste it in your React app!')
print(f'\n⚠️  Keep this tab open — closing it stops the server.')
print('=' * 60)

# Run the FastAPI server
uvicorn.run('api_server:app', host='0.0.0.0', port=8000)